# Generate Prompt Examples for Paper

This notebook generates prompt examples in various formats for use in academic papers.
It takes the first 5 senses and creates properly formatted prompts that can be saved as text files.

In [22]:
from data_loader import load_sense_repo
from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
from config import ANNOTATION_CHUNKS, DATA_DIR
from process_senses import default_build_senses_block
from preprocessing import (
    get_mwe_tokens, mark_mwe, get_mwe_filtered_senses, get_token_senses,
    mark_token, token_is_content_word, token_is_not_in_ne_not_in_filter, token_is_not_in_mwe
)
from pathlib import Path

# Load sense repository
senses_df = load_sense_repo()

# Load first chunk of data
chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
chunk_begin, chunk_end, selected_path = chunks[0]
print(f"Loading data from: {selected_path.name}")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

# Take only first 5 sentences
sentences = sentences[:5]
print(f"Loaded {len(sentences)} sentences for prompt generation")

Loading data from: sr-elexis-WSD_0001_0500.tsv
Loaded 5 sentences for prompt generation


## System Message Template

In [23]:
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW_SENSE'; nikada ne smete izmišljati druge ID-ove.
"""

print("System Message:")
print("=" * 80)
print(system_message.strip())

System Message:
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW_SENSE'; nikada ne smete izmišljati druge ID-ove.


## Generate Prompts for Each Sentence

In [24]:
# LaTeX escape helper function
def escape_latex(text):
    """Escape special LaTeX characters."""
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

print("LaTeX helper function loaded")

LaTeX helper function loaded


In [25]:
from config import L_LEMMA, S_ID, S_DEFINITION

# Collect all prompt examples
prompt_examples = []

for idx, sent in enumerate(sentences, 1):
    # Process MWEs first
    for mwe in sent.mwes:
        if mwe.lemma == "*":
            continue
        
        marked_sentence = mark_mwe(mwe, sent, start_mark="<b>", end_mark="</b>")
        senses_df_slice = get_mwe_filtered_senses(mwe, senses_df)
        
        if len(senses_df_slice) > 0:
            sense_list = [{
                'senseID': row[S_ID],
                'definition': row[S_DEFINITION]
            } for _, row in senses_df_slice.iterrows()]
            
            senses_block = default_build_senses_block(sense_list)
            
            prompt_examples.append({
                'sentence_num': idx,
                'token_id': f"MWE_{mwe.lemma}",
                'word': mwe.lemma,
                'sentence': marked_sentence,
                'senses_block': senses_block,
                'num_senses': len(sense_list)
            })
    
    # Process individual tokens
    for token in sent.tokens:
        if not token_is_content_word(token):
            continue
        if not token_is_not_in_ne_not_in_filter(token):
            continue
        if not token_is_not_in_mwe(token):
            continue
        
        token_senses = get_token_senses(token, sent, senses_df)
        
        if len(token_senses) > 0:
            marked_sentence = mark_token(token, sent, start_mark="<b>", end_mark="</b>")
            
            sense_list = [{
                'senseID': row[S_ID],
                'definition': row[S_DEFINITION]
            } for _, row in token_senses.iterrows()]
            
            senses_block = default_build_senses_block(sense_list)
            
            prompt_examples.append({
                'sentence_num': idx,
                'token_id': token.token_index,
                'word': token.layers.get(L_LEMMA, token.text),
                'sentence': marked_sentence,
                'senses_block': senses_block,
                'num_senses': len(sense_list)
            })

print(f"Generated {len(prompt_examples)} prompt examples")

Generated 41 prompt examples


## Format 1: Full Prompt (System + User Message)

In [27]:
def format_full_prompt(example, system_msg):
    """Format a complete LaTeX-friendly prompt with system and user messages."""
    # First escape LaTeX, then replace <b> tags with \textbf{}
    sentence = escape_latex(example['sentence'])
    sentence = sentence.replace('<b>', r'\textbf{').replace('</b>', '}')
    
    word = escape_latex(example['word'])
    senses_block = escape_latex(example['senses_block'])
    system_escaped = escape_latex(system_msg.strip())
    
    full_prompt = f"""\\textbf{{SYSTEM MESSAGE:}}

{system_escaped}

\\hrule

\\textbf{{USER MESSAGE:}}

Kontekst rečenice (ciljna reč je označena kao bold):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""
    return full_prompt.strip()

# Generate and display first example
if prompt_examples:
    example = prompt_examples[0]
    full = format_full_prompt(example, system_message)
    print(f"Example 1 (Sentence {example['sentence_num']}, Word: {example['word']}):")
    print("=" * 80)
    print(full)

Example 1 (Sentence 1, Word: u skladu sa):
\textbf{SYSTEM MESSAGE:}

Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

\{\{
  "sense\_id": "<jedan od ponuđenih ID-jeva ili 'NEW\_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW\_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
\}\}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW\_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense\_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW\_SENSE'; nikada ne smete izmišljati druge ID-ove.

\hrule

\textbf{USER MESSAGE:}

Kontekst rečenice (ciljna reč je označena kao bold):
"U petoj sezoni program je održan \textbf{u} \textbf{s

## Format 2: User Prompt Only (for appendix)

In [28]:
def format_user_prompt_only(example):
    """Format LaTeX-friendly user prompt only."""
    # First escape LaTeX, then replace <b> tags with \textbf{}
    sentence = escape_latex(example['sentence'])
    sentence = sentence.replace('<b>', r'\textbf{').replace('</b>', '}')
    
    word = escape_latex(example['word'])
    senses_block = escape_latex(example['senses_block'])
    
    user_prompt = f"""
Kontekst rečenice (ciljna reč je označena kao bold):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""
    return user_prompt.strip()

# Generate and display
if prompt_examples:
    example = prompt_examples[0]
    user_only = format_user_prompt_only(example)
    print(f"User Prompt Only - Example 1:")
    print("=" * 80)
    print(user_only)

User Prompt Only - Example 1:
Kontekst rečenice (ciljna reč je označena kao bold):
"U petoj sezoni program je održan \textbf{u} \textbf{skladu} \textbf{sa} novim konceptom: jedan mladi fudbaler iz svake zemlje izabran je kao njen predstavnik."

Ciljna reč: "u skladu sa"

Lista mogućih značenja:
1. ID: LLM-0288 — U harmoniji sa; prema nečemu; saglasno nečemu.


## Format 3: Compact Format (for tables)

In [29]:
def format_compact(example, system_msg):
    """Format LaTeX-friendly compact version with system + user prompt."""
    # First escape LaTeX, then replace <b> tags with \textbf{}
    sentence = escape_latex(example['sentence'])
    sentence = sentence.replace('<b>', r'\textbf{').replace('</b>', '}')
    
    word = escape_latex(example['word'])
    senses_block = escape_latex(example['senses_block'])
    system_escaped = escape_latex(system_msg.strip())
    
    compact = f"""
\\textbf{{System:}} {system_escaped}

\\textbf{{Sentence:}} {sentence}
\\textbf{{Target:}} {word}
\\textbf{{Senses ({example['num_senses']}):}}
{senses_block}
"""
    return compact.strip()

# Generate and display
if prompt_examples:
    example = prompt_examples[0]
    compact = format_compact(example, system_message)
    print(f"Compact Format - Example 1:")
    print("=" * 80)
    print(compact)

Compact Format - Example 1:
\textbf{System:} Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

\{\{
  "sense\_id": "<jedan od ponuđenih ID-jeva ili 'NEW\_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW\_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
\}\}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW\_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense\_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW\_SENSE'; nikada ne smete izmišljati druge ID-ove.

\textbf{Sentence:} U petoj sezoni program je održan \textbf{u} \textbf{skladu} \textbf{sa} novim konceptom: jedan mladi fudbaler iz svake zemlje izabran je kao nje

## Format 4: LaTeX Enumerate Format

In [34]:
def format_latex_enumerate(example, system_msg):
    """Format with LaTeX enumerate for senses, including system + user prompt."""
    # First escape LaTeX, then replace <b> tags with \textbf{}
    sentence = escape_latex(example['sentence'])
    sentence = sentence.replace('<b>', r'\textbf{').replace('</b>', '}')
    
    word = escape_latex(example['word'])
    system_escaped = escape_latex(system_msg.strip())
    
    latex_format = f"""
\\textit{{System Message:}}

{system_escaped}

\\textit{{User Message:}}

\\textbf{{Kontekst:}} {sentence}

\\textbf{{Ciljna reč:}} {word}

\\textbf{{Moguća značenja:}}
\\begin{{enumerate}}
"""
    
    # Parse senses and format as enumerate items
    senses_lines = example['senses_block'].split('\n')
    for line in senses_lines:
        if line.strip():
            # Remove the number prefix since enumerate handles it
            parts = line.split('. ', 1)
            if len(parts) > 1:
                line_content = escape_latex(parts[1])
            else:
                line_content = escape_latex(line)
            latex_format += f"    \\item {line_content}\n"
    
    latex_format += "\\end{enumerate}"
    
    return latex_format.strip()

# Generate and display
if prompt_examples:
    example = prompt_examples[0]
    latex = format_latex_enumerate(example, system_message)
    print(f"LaTeX Enumerate Format - Example 1:")
    print("=" * 80)
    print(latex)

LaTeX Enumerate Format - Example 1:
\textit{System Message:}

Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

\{\{
  "sense\_id": "<jedan od ponuđenih ID-jeva ili 'NEW\_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW\_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
\}\}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW\_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense\_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW\_SENSE'; nikada ne smete izmišljati druge ID-ove.

\textit{User Message:}

\textbf{Kontekst:} U petoj sezoni program je održan \textbf{u} \textbf{skladu} \textbf{sa} novim konceptom: jedan mladi fu

## Save All Examples to Text Files

In [35]:
# Create output directory for prompt examples
output_dir = Path('output/prompt_examples')
output_dir.mkdir(parents=True, exist_ok=True)

# Save each format type for all examples
for idx, example in enumerate(prompt_examples, 1):
    base_name = f"example_{idx:02d}_sent{example['sentence_num']}_word_{example['word']}"
    
    # Format 1: Full prompt (LaTeX-friendly with system + user)
    full = format_full_prompt(example, system_message)
    with open(output_dir / f"{base_name}_full.txt", 'w', encoding='utf-8') as f:
        f.write(full)
    
    # Format 2: User prompt only (LaTeX-friendly)
    user = format_user_prompt_only(example)
    with open(output_dir / f"{base_name}_user.txt", 'w', encoding='utf-8') as f:
        f.write(user)
    
    # Format 3: Compact (LaTeX-friendly with system + user)
    compact = format_compact(example, system_message)
    with open(output_dir / f"{base_name}_compact.txt", 'w', encoding='utf-8') as f:
        f.write(compact)
    
    # Format 4: LaTeX enumerate (with system + user)
    latex = format_latex_enumerate(example, system_message)
    with open(output_dir / f"{base_name}_latex.txt", 'w', encoding='utf-8') as f:
        f.write(latex)

print(f"\nSaved {len(prompt_examples)} examples in 4 formats to: {output_dir}")
print(f"Total files created: {len(prompt_examples) * 4}")


Saved 41 examples in 4 formats to: output\prompt_examples
Total files created: 164


## Create Summary File

In [32]:
# Create a summary file listing all examples
summary_file = output_dir / "_summary.txt"

with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("PROMPT EXAMPLES SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total examples generated: {len(prompt_examples)}\n")
    f.write(f"Generated from first {len(sentences)} sentences\n\n")
    
    f.write("Examples:\n")
    f.write("-" * 80 + "\n")
    
    for idx, example in enumerate(prompt_examples, 1):
        f.write(f"\n{idx}. Sentence {example['sentence_num']}, Word: '{example['word']}' ({example['num_senses']} senses)\n")
        f.write(f"   Files: example_{idx:02d}_sent{example['sentence_num']}_word_{example['word']}_*.txt\n")
    
    f.write("\n" + "=" * 80 + "\n")
    f.write("\nFile Format Types (all LaTeX-friendly):\n")
    f.write("  *_full.txt    - Complete prompt with system and user messages (LaTeX)\n")
    f.write("  *_user.txt    - User message only (LaTeX)\n")
    f.write("  *_compact.txt - Compact format with system+user (LaTeX)\n")
    f.write("  *_latex.txt   - LaTeX enumerate format with system+user\n")

print(f"\nSummary saved to: {summary_file}")


Summary saved to: output\prompt_examples\_summary.txt


## Display Summary Statistics

In [33]:
print("\n" + "=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)
print(f"\nTotal prompt examples: {len(prompt_examples)}")
print(f"From sentences: {len(sentences)}")
print(f"\nSense distribution:")
for num_senses in sorted(set(ex['num_senses'] for ex in prompt_examples)):
    count = sum(1 for ex in prompt_examples if ex['num_senses'] == num_senses)
    print(f"  {num_senses} senses: {count} examples")

print(f"\nOutput directory: {output_dir.absolute()}")
print(f"\nYou can now review the text files and select the best examples for your paper.")


GENERATION COMPLETE

Total prompt examples: 41
From sentences: 5

Sense distribution:
  1 senses: 9 examples
  2 senses: 7 examples
  3 senses: 11 examples
  4 senses: 1 examples
  5 senses: 5 examples
  6 senses: 3 examples
  7 senses: 4 examples
  11 senses: 1 examples

Output directory: e:\Github\LexiSense-SR\output\prompt_examples

You can now review the text files and select the best examples for your paper.
